# 5. Integer linear programme

Lecture: *Week 3 - Linear Programming 2*, slide 24.

*Original script: `5 Source Files/Python/intlinprog_pyomo.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

In [ ]:
from pyomo.environ import *

In [ ]:
# Create a concrete model
model = ConcreteModel()

In [ ]:
# Define the integer decision variables with lower and upper bounds
model.x = Var([0, 1], domain=NonNegativeIntegers)

In [ ]:
# Define the objective function coefficients as a list
f = [2, 3.5]

In [ ]:
# Define the objective function
model.obj = Objective(expr=sum(f[i] * model.x[i] for i in [0, 1]), sense=maximize)

In [ ]:
# Define the inequality constraints
c1 = 3 * model.x[0] + 2 * model.x[1] <= 14
c2 = 2 * model.x[0] + 3 * model.x[1] <= 12.5
c3 = -model.x[0] + model.x[1] <= 1

In [ ]:
# Add the constraints to the model
model.c1 = Constraint(expr=c1)
model.c2 = Constraint(expr=c2)
model.c3 = Constraint(expr=c3)

In [ ]:
# Solve the integer optimization problem
solver = SolverFactory('appsi_highs')
results = solver.solve(model, tee=False)  # Set tee=True to display solver output

In [ ]:
# Check the solver status and print the results
if results.solver.status == SolverStatus.ok:
    if results.solver.termination_condition == TerminationCondition.optimal:
        print("Optimal solution found.")
        print("Objective value:", model.obj())
        print("Decision variables:")
        for i in [0, 1]:
            print(f"x[{i}] = {model.x[i].value}")
    else:
        print("Solver terminated with a non-optimal solution.")
else:
    print("Solver did not find a solution. Status:", results.solver.status)
    
    